In [2]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

oilp = pd.read_csv('data/wti_oil.csv')
oilp = oilp.rename(columns={'DCOILWTICO': 'price'})

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
oilp['DATE'] = pd.to_datetime(oilp['DATE']) - pd.offsets.MonthEnd(1)

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

oilp['pc_1q'] = np.log(oilp['price'].shift(1) / oilp['price'].shift(2))
oilp['pc_2q'] = np.log(oilp['price'].shift(2) / oilp['price'].shift(3))

In [7]:
###############################################
                ## Controls ##
###############################################

def inf_control(df, v, end_date):
    v['lt_3'] = ((v['t']/400 + 1)*(v['t1']/400 + 1)*(v['t2']/400 + 1)*(v['t3']/400 + 1)-1)
    v['lt_3'] = v['lt_3'].shift(1)
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    v = v.loc[(v['DATE'] >= '1981-12-31') & (v['DATE'] <= end_date)]
    L_pi = v['lt_3']
    revisions = df[f'r_t3']
    x = np.column_stack((L_pi, revisions))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    initial = sm.OLS(errors, x).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


def oil_control(df, oil, end_date):
    df = df.loc[(df['DATE'] >= '1986-06-30') & (df['DATE'] <= end_date)]
    oil = oil.loc[(oil['DATE'] >= '1986-06-30') & (oil['DATE'] <= end_date)]
    revisions = df[f'r_t3']
    L_oil = oil['pc_1q']
    x = np.column_stack((L_oil, revisions))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    initial = sm.OLS(errors, x).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


date = '2020-06-30'
regs = [inf_control(mean_spf_trim, vintage_trim, date), oil_control(mean_spf_trim, oilp, date)]

regs[1].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.145
Model:                            OLS   Adj. R-squared:                  0.132
Method:                 Least Squares   F-statistic:                     9.733
Date:                Wed, 08 May 2024   Prob (F-statistic):           0.000113
Time:                        16:58:10   Log-Likelihood:                 451.41
No. Observations:                 137   AIC:                            -896.8
Df Residuals:                     134   BIC:                            -888.1
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0019      0.001     -1.506      0.134      -0.004       0.001
x1             0.0338      0.008      4.363      0.000       0.018       0.049
x2            -0.9675      0.324     -2.984      0.003      -1.609      -0.326
==============================================================================
Omnibus:                        6.893   Durbin-Watson:                   0.750
Prob(Omnibus):                  0.032   Jarque-Bera (JB):               11.715
Skew:                          -0.096   Prob(JB):                      0.00286
Kurtosis:                       4.420   Cond. No.                         360.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [9]:
###############################################
         ## Heterogeneity in Priors ##
###############################################

### Individual Forecasts ###
def E3R2_PLD(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    revisions = df['r_t2']
    revisions = sm.add_constant(revisions)
    errors = df[f'e_t3']
    regs = sm.OLS(errors, revisions).fit(cov_type='cluster', cov_kwds = {"groups":df['ID']})
    return regs


### Mean Forecasts ###
def IV_OIL(df, oil, v, end_date):
    v['lt_3']= ((v['t']/400 + 1)*(v['t1']/400 + 1)*(v['t2']/400 + 1)*(v['t3']/400 + 1)-1)
    L_pi = v['lt_3'].shift(1)
    df = df.loc[(df['DATE'] >= '1986-09-30') & (df['DATE'] <= end_date)]
    oil = oil.loc[(oil['DATE'] >= '1986-09-30') & (oil['DATE'] <= end_date)]
    v = v.loc[(v['DATE'] >= '1986-09-30') & (v['DATE'] <= end_date)]
    L_pi = v['lt_3']
    revisions = df[f'r_t3']
    x = np.column_stack((revisions, L_pi))
    x = sm.add_constant(x)
    errors = df[f'e_t3']
    L1_oil = oil['pc_1q']
    L2_oil = oil['pc_2q']
    w = np.column_stack((L_pi, L1_oil, L2_oil))
    w = sm.add_constant(w)
    reg = IV2SLS(errors, x, instrument = w).fit()
    return reg
    
    
date = '2020-06-30'
regs = [E3R2_PLD(ind_spf_trim, date), IV_OIL(mean_spf_trim, oilp, vintage_trim, date)]

regs[1].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          IV2SLS Regression Results                           
==============================================================================
Dep. Variable:                   e_t3   R-squared:                      -0.074
Model:                         IV2SLS   Adj. R-squared:                 -0.090
Method:                     Two Stage   F-statistic:                     2.600
                        Least Squares   Prob (F-statistic):             0.0780
Date:                Wed, 08 May 2024                                         
Time:                        17:00:21                                         
No. Observations:                 136                                         
Df Residuals:                     133                                         
Df Model:                           2                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0017      0.004      0.429      0.669      -0.006       0.009
x1             0.6638      0.312      2.125      0.035       0.046       1.282
x2            -0.0049      0.008     -0.607      0.545      -0.021       0.011
==============================================================================
Omnibus:                       16.241   Durbin-Watson:                   0.830
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               40.236
Skew:                          -0.400   Prob(JB):                     1.83e-09
Kurtosis:                       5.542   Cond. No.                         295.
==============================================================================
"""